In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

import ipywidgets as widgets
from IPython.display import display, clear_output

# Dataset con mensajes e intenciones
data = {
    'mensaje': [
        'hola', 'buenos días', 'qué tal',
        'a qué hora abren', 'cuál es su horario', 'están abiertos hoy',
        'dónde están ubicados', 'cómo llego al local', 'dame la dirección',
        'gracias', 'hasta luego', 'nos vemos'
    ],
    'intencion': [
        'saludo', 'saludo', 'saludo',
        'horario', 'horario', 'horario',
        'ubicacion', 'ubicacion', 'ubicacion',
        'despedida', 'despedida', 'despedida'
    ]
}
df = pd.DataFrame(data)

def entrenar_modelo(df):
    y_encoded = le.fit_transform(df['intencion'])
    modelo = make_pipeline(CountVectorizer(), MultinomialNB())
    modelo.fit(df['mensaje'], y_encoded)
    return modelo

le = LabelEncoder()
modelo = entrenar_modelo(df)

def responder_y_predecir(mensaje_usuario):
    pred = modelo.predict([mensaje_usuario])[0]
    intencion_predicha = le.inverse_transform([pred])[0]

    respuestas = {
        'saludo': "👋 ¡Hola! ¿En qué puedo ayudarte hoy?",
        'horario': "🕘 Nuestro horario es de lunes a viernes de 9 a.m. a 6 p.m.",
        'ubicacion': "📍 Estamos en Av. Principal #123, cerca del parque central.",
        'despedida': "👋 ¡Gracias por escribirnos! Que tengas un excelente día."
    }

    respuesta = respuestas.get(intencion_predicha, "🤖 Lo siento, no entendí tu mensaje.")
    return respuesta, intencion_predicha

# Widgets principales
entrada = widgets.Text(placeholder='Escribe tu mensaje...', description='Tú:', layout=widgets.Layout(width='100%'))
boton = widgets.Button(description="Enviar")
salida = widgets.Output()

# Corrección de intención
intencion_dropdown = widgets.Dropdown(options=df['intencion'].unique(), description="Intención correcta:")
boton_corregir = widgets.Button(description="Corregir")
correccion_box = widgets.VBox([])

# Variables temporales
ultimo_mensaje = ''

# Función para manejar envío del mensaje
def cuando_envia(b):
    global ultimo_mensaje
    mensaje = entrada.value.strip()
    if mensaje:
        respuesta, intencion_predicha = responder_y_predecir(mensaje)
        ultimo_mensaje = mensaje
        with salida:
            clear_output()
            print(f"Tú: {mensaje}")
            print(f"Chatbot: {respuesta}")
            print(f"🤖 Intención detectada: {intencion_predicha}")
        correccion_box.children = [widgets.Label("¿La intención detectada fue incorrecta?"), intencion_dropdown, boton_corregir]
    entrada.value = ''

# Función para corregir y reentrenar
def cuando_corrige(b):
    global df, modelo
    intencion_correcta = intencion_dropdown.value
    df = pd.concat([df, pd.DataFrame({'mensaje': [ultimo_mensaje], 'intencion': [intencion_correcta]})], ignore_index=True)
    modelo = entrenar_modelo(df)
    with salida:
        print("✅ Intención corregida y modelo actualizado.")
    correccion_box.children = []

# Asociar eventos
boton.on_click(cuando_envia)
boton_corregir.on_click(cuando_corrige)

# Mostrar interfaz
display(widgets.VBox([entrada, boton, salida, correccion_box]))